<a href="https://colab.research.google.com/github/CreatorPoints/PhotonCoreV2/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Super Wav2Lip

This Colab project is based on [Wav2Lip-GFPGAN](https://github.com/ajay-sainy/Wav2Lip-GFPGAN), but updates the requirements.txt (to function properly) and updates Colab file for ease of use.

In [1]:
!sed -i 's/np.complex/complex/g' /usr/local/lib/python3.12/dist-packages/librosa/core/constantq.py 2>/dev/null
!sed -i 's/np.float/float/g' /usr/local/lib/python3.12/dist-packages/librosa/util/utils.py 2>/dev/null
!sed -i 's/np.float/float/g' /usr/local/lib/python3.12/dist-packages/librosa/core/spectrum.py 2>/dev/null
import sys, numpy as np

# Monkeypatch missing NumPy 1.x legacy aliases directly into numpy
np.float = float
np.complex = complex
np.int = int
np.bool = bool
np.float32 = np.float32
np.complex128 = np.complex128

# Re-install librosa cleanly without broken sed edits
!pip install --quiet --upgrade librosa
print("✅ NumPy monkeypatch applied & librosa restored!")
import site, glob, os

# Find all installed librosa files in Colab's python site-packages
site_packages = site.getsitepackages()[0]
librosa_files = glob.glob(os.path.join(site_packages, "librosa", "**", "*.py"), recursive=True)

# Replace deprecated NumPy 1.x aliases with native Python types
for filepath in librosa_files:
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()

    # Direct replacement of obsolete attributes
    content = content.replace("np.complex", "complex")
    content = content.replace("np.float", "float")
    content = content.replace("np.int", "int")
    content = content.replace("np.bool", "bool")

    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("✅ System-wide librosa patch applied successfully!")

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/yapf-0.43.0-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/tb_nightly-2.21.0a20251023-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
✅ NumPy monkeypatch applied & librosa restored!
✅ System-wide librosa patch applied successfully!


## 1. Installation

Run this block to install the necessary dependencies.

In [2]:
!git clone https://github.com/indianajson/wav2lip-HD.git
basePath = "/content/wav2lip-HD"
%cd {basePath}

wav2lipFolderName = 'Wav2Lip-master'
gfpganFolderName = 'GFPGAN-master'
wav2lipPath = basePath + '/' + wav2lipFolderName
gfpganPath = basePath + '/' + gfpganFolderName

!wget 'https://www.adrianbulat.com/downloads/python-fan/s3fd-619a316812.pth' -O {wav2lipPath}'/face_detection/detection/sfd/s3fd.pth'

!wget 'https://huggingface.co/Nekochu/Wav2Lip/resolve/main/wav2lip_gan.pth' -O {wav2lipPath}'/checkpoints/wav2lip_gan.pth'
#!wget 'https://iiitaphyd-my.sharepoint.com/personal/radrabha_m_research_iiit_ac_in/_layouts/15/download.aspx?share=EdjI7bZlgApMqsVoEUUXpLsBxqXbn5z8VTmoxp55YNDcIA' -O {wav2lipPath}'/checkpoints/wav2lip_gan.pth'
#!wget 'https://iiitaphyd-my.sharepoint.com/:u:/g/personal/radrabha_m_research_iiit_ac_in/Eb3LEzbfuKlJiR600lQWRxgBIY27JZg80f7V9jtMfbNDaQ?e=TBFBVW' -O {wav2lipPath}'/checkpoints/wav2lip.pth'

!gdown https://drive.google.com/uc?id=1fQtBSYEyuai9MjBOF8j7zZ4oQ9W2N64q --output {wav2lipPath}'/checkpoints/'

!pip install -r requirements.txt
!pip install -U librosa==0.8.1 # The process will fail without downgrading librosa
!mkdir inputs

!cd $gfpganFolderName && python setup.py develop
!wget https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth -P {gfpganFolderName}'/experiments/pretrained_models'

%cd {basePath}

from IPython.display import clear_output
clear_output()

print("Installation complete.")

Installation complete.


## 2. Synchronize Video and Speech

In [5]:
import os, sys, glob, site

# Define paths
basePath = "/content/wav2lip-HD"
wav2lipFolderName = "/content/wav2lip-HD/Wav2Lip-master"

outputPath = basePath + '/outputs'
inputAudio = 'bruh.mp3' #@param{type:"string"}
inputAudioPath = basePath + '/inputs/' + inputAudio
inputVideo = 'guy.mp4' #@param{type:"string"}
inputVideoPath = basePath + '/inputs/' + inputVideo
lipSyncedOutputPath = basePath + '/outputs/result.mp4'
model = "wav2lip" #@param ["wav2lip", "wav2lip_gan"] {type:"string"}

if not os.path.exists(outputPath):
    os.makedirs(outputPath)

# 1. Patch librosa files on disk to fix Python 3.12 / NumPy 2.x attribute errors
site_packages = site.getsitepackages()[0]
librosa_files = glob.glob(os.path.join(site_packages, "librosa", "**", "*.py"), recursive=True)
for filepath in librosa_files:
    try:
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        content = content.replace("np.complex", "complex").replace("np.float", "float").replace("np.int", "int")
        with open(filepath, "w", encoding="utf-8") as f:
            f.write(content)
    except Exception:
        pass

# 2. Patch Wav2Lip's local audio.py file
audio_py = os.path.join(wav2lipFolderName, "audio.py")
if os.path.exists(audio_py):
    with open(audio_py, "r", encoding="utf-8") as f:
        c = f.read()
    c = c.replace("np.complex", "complex").replace("np.float", "float").replace("np.int", "int")
    with open(audio_py, "w", encoding="utf-8") as f:
        f.write(c)

# 3. Change directory and run inference directly via sys.argv
os.chdir(wav2lipFolderName)
sys.argv = [
    'inference.py',
    '--checkpoint_path', f'checkpoints/{model}.pth',
    '--face', inputVideoPath,
    '--audio', inputAudioPath,
    '--outfile', lipSyncedOutputPath
]

print("🚀 Running Wav2Lip synthesis...")
exec(open('inference.py').read())
print("✅ DONE! Check /content/wav2lip-HD/outputs/result.mp4")

🚀 Running Wav2Lip synthesis...


NameError: name 'float32' is not defined

## 3. Boost the Resolution of the Synthesized Video



In [ ]:
import cv2
from tqdm import tqdm
from os import path

import os

inputVideoPath = outputPath+'/result.mp4'
unProcessedFramesFolderPath = outputPath+'/frames'

if not os.path.exists(unProcessedFramesFolderPath):
  os.makedirs(unProcessedFramesFolderPath)

vidcap = cv2.VideoCapture(inputVideoPath)
numberOfFrames = int(vidcap.get(cv2.CAP_PROP_FRAME_COUNT))
fps = vidcap.get(cv2.CAP_PROP_FPS)
print("FPS: ", fps, "Frames: ", numberOfFrames)

for frameNumber in tqdm(range(numberOfFrames)):
    _,image = vidcap.read()
    cv2.imwrite(path.join(unProcessedFramesFolderPath, str(frameNumber).zfill(4)+'.jpg'), image)


!cd $gfpganFolderName && \
  python inference_gfpgan.py -i $unProcessedFramesFolderPath -o $outputPath -v 1.3 -s 2 --only_center_face --bg_upsampler None

import os
restoredFramesPath = outputPath + '/restored_imgs/'
processedVideoOutputPath = outputPath

if not os.path.exists(restoredFramesPath):
  os.makedirs(restoredFramesPath)

dir_list = os.listdir(restoredFramesPath)
dir_list.sort()

import cv2
import numpy as np

#Get FPS of original video for writer
inputVideoPath = outputPath+'/result.mp4'
vidcap = cv2.VideoCapture(inputVideoPath)
fps = vidcap.get(cv2.CAP_PROP_FPS)
print("The video is "+str(fps)+" FPS.")

batch = 0
batchSize = 1300
from tqdm import tqdm
for i in tqdm(range(0, len(dir_list), batchSize)):
  img_array = []
  start, end = i, i+batchSize
  print("processing ", start, end, end="\r")
  for filename in  tqdm(dir_list[start:end]):
      filename = restoredFramesPath+filename;
      img = cv2.imread(filename)
      if img is None:
        continue
      height, width, layers = img.shape
      size = (width,height)
      img_array.append(img)
  out = cv2.VideoWriter(processedVideoOutputPath+'/output_'+str(batch).zfill(4)+'.mp4',cv2.VideoWriter_fourcc(*'DIVX'), fps, size)
  batch = batch + 1

  for i in range(len(img_array)):
    out.write(img_array[i])
  out.release()

from IPython.display import clear_output
clear_output()

print("Video upscaling complete.")

## 4. Clear Cached Files

Run this block once you've downloaded your final video file. This will empty /inputs and /outputs, so you can start again, fresh.


In [ ]:
%cd /content/wav2lip-HD/

#@markdown Choose whether to remove both inputs and outputs, or just one of the two. You may want to preserve inputs if you are only changing one of the two inputs.

removeInputs = True #@param {type:"boolean"}
removeOutputs = True #@param {type:"boolean"}

if removeInputs == True:
  %rm inputs/*
if removeOutputs == True:
  %rm outputs/frames/*
  %rm outputs/restored_imgs/*
  %rm outputs/*


from IPython.display import clear_output
clear_output()

print("Cleared cached files.")
